## Validate the SPDI columns
- Iterate over the output directory and collect a list of all metadata files

In [69]:
import ast
from importlib import reload
import os
import pandas as pd
import sys
import subprocess

sys.path.append('../helpful_functions')
import helpful_functions as hf
reload(hf)



<module 'helpful_functions' from '/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/../helpful_functions/helpful_functions.py'>

In [70]:
# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'


interesting_columns = [col_name, col_sequence, col_category, col_class, col_source, col_ref,
                       col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]

### Validate SPDI

In [ ]:
def list_metadata_files_in_subdirectories(directory):
    """Returns all the files in the subdirectories of a given directory"""
    file_list = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if 'metadata.tsv.gz' in file:
                file_list.append(os.path.join(root, file))
                # print(os.path.join(root, file))
    return file_list

# Usage
directory_path = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format'
file_list = list_metadata_files_in_subdirectories(directory_path)

import ast

# Function to safely evaluate string representations of lists
def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x

# list columns col_variant_class, col_variant_pos, col_SPDI, col_allele,
list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]

def verify_spdi_result(verified_spdi_file, all_spdi_df, error_file):
    """"""
    verified_spdis = pd.read_csv(verified_spdi_file, header=None,sep="\t")
    verified_spdis.columns = ['input_spdi', 'verified_spdi']
    # print the rows where a second column is not in the file
    verified_spdis['is_verified'] = verified_spdis.apply(lambda row: row['input_spdi'] == row['verified_spdi'], axis=1)
    if verified_spdis.loc[~verified_spdis['is_verified']].shape[0] > 0:
        print(f'Found problem in SPDI. The output can be found in {error_file}.')
    verified_spdis.loc[~verified_spdis['is_verified']].to_csv(error_file, header=False, index=None, sep="\t")
    # check which SPDIs are missing in the second column of the resulting file => rerun them a second time
    all_spdis = set(all_spdi_df[col_SPDI].to_list())
    matchable_spdis = set(verified_spdis['verified_spdi'].to_list())
    # plot the list or let the script run again
    not_matchable_spdis = all_spdis - matchable_spdis
    return not_matchable_spdis

In [ ]:
variant_groups = ['​GC_Liang​', 'GC_Mohlke​', 'GC_Atrial_fib​']

In [72]:
file_list

['/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Mohlke/GC_Mohlke.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/C_positive_heart_CAD/C_positive_heart_CAD.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/C_positive_heart_AB/C_positive_heart_AB.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Glut_Chengyu/GC_Glut_Chengyu.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/C_negative_heart_MK/C_negative_heart_MK.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_Liang/GC_Liang.metadata.tsv.gz',
 '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/GC_GABA_Chengyu/GC_GABA_Chengyu.metadata.tsv.gz',
 '/data/cephf

In [73]:
subfolder_list = [file.split("/")[-2] for file in file_list]

In [81]:
file = file_list[0]
file = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/C_positive_heart_AB/C_positive_heart_AB.metadata.tsv.gz'
file = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/MK/MK.metadata.tsv.gz'

group_name = file.split('/')[-1].split(".metadata")[0]
print(group_name)
# load metadata file
metadata_file = pd.read_csv(file, sep="\t")

# Apply the safe_eval function to the specified columns
for col in list_columns:
    metadata_file[col] = metadata_file[col].apply(safe_eval)

# extract the spdi column of the alternatives
alternatives_df = metadata_file.loc[metadata_file[col_allele].apply(hf.is_alternative)].copy() # 20
if alternatives_df.shape[0] == 0:
    print('No variants given.')
current_path = os.path.abspath(os.getcwd())
output_file = os.path.join(current_path, f'SPDI_{group_name}.csv')
verified_spdi_file = os.path.join(current_path, f'verified_SPDI_{group_name}.csv')
alternatives_df[['SPDI']].to_csv(output_file, header=False, index=None)
# run the spdi batch checking script
# Run a command and capture its output
script_name = "/home/kisa/coding/80K_MPRA/igvf_spdi_demo_mike/spdi_batch.py"
input_arg = f"-i {output_file}"
command = [
    "python",
    "/home/kisa/coding/80K_MPRA/igvf_spdi_demo_mike/spdi_batch.py",
    "-i", output_file,
    "-t", "SPDI"
]
with open(verified_spdi_file, "w") as verified_file_handle:
    result = subprocess.run(command, stdout=verified_file_handle, text=True)

spdi_error_file = os.path.join(current_path, f'error_SPDI_{group_name}.tsv')
not_matchable_spdis = verify_spdi_result(verified_spdi_file, alternatives_df, spdi_error_file)

if len(not_matchable_spdis) > 0:
    output_file_second = os.path.join(current_path, f'second_SPDI_{group_name}.csv')
    alternatives_df.loc[alternatives_df[col_SPDI].isin(not_matchable_spdis)][['SPDI']].to_csv(output_file_second, header=False, index=None)
    command = [
    "python",
    "/home/kisa/coding/80K_MPRA/igvf_spdi_demo_mike/spdi_batch.py",
    "-i", output_file_second,
    "-t", "SPDI"
    ]
    verified_spdi_file_second = os.path.join(current_path, f'second_verified_SPDI_{group_name}.csv')
    with open(verified_spdi_file_second, "w") as verified_file_handle:
        result = subprocess.run(command, stdout=verified_file_handle, text=True)
    spdi_error_file_second = os.path.join(current_path, f'second_error_SPDI_{group_name}.tsv')
    verify_spdi_result(verified_spdi_file_second, alternatives_df, spdi_error_file_second)

MK


ValueError: Length mismatch: Expected axis has 1 elements, new values have 2 elements

In [82]:
verified_spdi_file

'/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/verified_SPDI_MK.csv'

In [ ]:
file = file_list[0]
for file in file_list:
    group_name = file.split('/')[-1].split(".metadata")[0]
    print(group_name)
    # load metadata file
    metadata_file = pd.read_csv(file, sep="\t")

    # Apply the safe_eval function to the specified columns
    for col in list_columns:
        metadata_file[col] = metadata_file[col].apply(safe_eval)

    # extract the spdi column of the alternatives
    alternatives_df = metadata_file.loc[metadata_file[col_allele].apply(hf.is_alternative)].copy() # 20
    current_path = os.path.abspath(os.getcwd())
    output_file = os.path.join(current_path, f'SPDI_{group_name}.csv')
    verified_spdi_file = os.path.join(current_path, f'verified_SPDI_{group_name}.csv')
    alternatives_df[['SPDI']].to_csv(output_file, header=False, index=None)
    # run the spdi batch checking script
    # Run a command and capture its output
    script_name = "/home/kisa/coding/80K_MPRA/igvf_spdi_demo_mike/spdi_batch.py"
    input_arg = f"-i {output_file}"
    command = [
        "python",
        "/home/kisa/coding/80K_MPRA/igvf_spdi_demo_mike/spdi_batch.py",
        "-i", output_file,
        "-t", "SPDI"
    ]
    with open(verified_spdi_file, "w") as verified_file_handle:
        result = subprocess.run(command, stdout=verified_file_handle, text=True)

    spdi_error_file = os.path.join(current_path, f'error_SPDI_{group_name}.tsv')
    not_matchable_spdis = verify_spdi_result(verified_spdi_file, alternatives_df, spdi_error_file)

    if len(not_matchable_spdis) > 0:
        output_file_second = os.path.join(current_path, f'second_SPDI_{group_name}.csv')
        alternatives_df.loc[alternatives_df[col_SPDI].isin(not_matchable_spdis)][['SPDI']].to_csv(output_file_second, header=False, index=None)
        command = [
        "python",
        "/home/kisa/coding/80K_MPRA/igvf_spdi_demo_mike/spdi_batch.py",
        "-i", output_file_second,
        "-t", "SPDI"
        ]
        verified_spdi_file_second = os.path.join(current_path, f'second_verified_SPDI_{group_name}.csv')
        with open(verified_spdi_file_second, "w") as verified_file_handle:
            result = subprocess.run(command, stdout=verified_file_handle, text=True)
        spdi_error_file_second = os.path.join(current_path, f'second_error_SPDI_{group_name}.tsv')
        verify_spdi_result(verified_spdi_file_second, alternatives_df, spdi_error_file_second)


GC_Mohlke


In [76]:
# verified_spdis = pd.read_csv(verified_spdi_file, header=None,sep="\t")
# verified_spdis.columns = ['input_spdi', 'verified_spdi']
# # print the rows where a second column is not in the file
# verified_spdis['is_verified'] = verified_spdis.apply(lambda row: row['input_spdi'] == row['verified_spdi'], axis=1)
# spdi_error_file = os.path.join(current_path, f'error_SPDI_{group_name}.tsv')
# if verified_spdis.loc[~verified_spdis['is_verified']].shape[0] > 0:
#     print(f'Found problem in SPDI. The output can be found in {spdi_error_file}.')
# verified_spdis.loc[~verified_spdis['is_verified']].to_csv(spdi_error_file, header=False, index=None, sep="\t")
# # check which SPDIs are missing in the second column of the resulting file => rerun them a second time
# all_spdis = set(alternatives_df[col_SPDI].to_list())
# matchable_spdis = set(verified_spdis['verified_spdi'].to_list())
# # plot the list or let the script run again
# not_matchable_spdis = all_spdis - matchable_spdis

In [ ]:
spdi_error_file = os.path.join(current_path, f'error_SPDI_{group_name}.tsv')
not_matchable_spdis = verify_spdi_result(verified_spdi_file, alternatives_df, spdi_error_file)

Found problem in SPDI. The output can be found in /home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/error_SPDI_GC_Mohlke.tsv.


In [78]:
if len(not_matchable_spdis) > 0:
    output_file_second = os.path.join(current_path, f'second_SPDI_{group_name}.csv')
    alternatives_df.loc[alternatives_df[col_SPDI].isin(not_matchable_spdis)][['SPDI']].to_csv(output_file_second, header=False, index=None)
    command = [
    "python",
    "/home/kisa/coding/80K_MPRA/igvf_spdi_demo_mike/spdi_batch.py",
    "-i", output_file_second,
    "-t", "SPDI"
    ]
    verified_spdi_file_second = os.path.join(current_path, f'second_verified_SPDI_{group_name}.csv')
    with open(verified_spdi_file_second, "w") as verified_file_handle:
        result = subprocess.run(command, stdout=verified_file_handle, text=True)
    spdi_error_file_second = os.path.join(current_path, f'second_error_SPDI_{group_name}.tsv')
    verify_spdi_result(verified_spdi_file_second, alternatives_df, spdi_error_file_second)

Found problem in SPDI. The output can be found in /home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/second_error_SPDI_GC_Mohlke.tsv.
